# F1 Strategy Dataset — Correlation Analysis

This notebook explores the relationship between each numeric feature and the target variable `PitNextLap`. We compute both Pearson and Spearman correlations.

## Data
- **File:** `data/f1_strategy_dataset_v4.csv`
- **Rows:** 101,371
- **Columns:** 16 (13 numeric features + 2 categorical + target)
- **Target:** `PitNextLap` (binary: 0 = no pit, 1 = pit)
- **Note:** 66 missing values in the `Compound` column (excluded from numeric correlation analysis)

## Approach
1. Load and inspect the data
2. Compute Pearson and Spearman correlations
3. Rank features by absolute correlation strength
4. Identify the most predictive features

## Correlation Analysis
We compute both Pearson (linear correlation) and Spearman (monotonic correlation) between each numeric feature and the target.

## Key Findings

1. **TyreLife** is the strongest predictor (Pearson: 0.266, Spearman: 0.139).
2. **LapNumber** is the second strongest (Pearson: 0.160, Spearman: 0.084).
3. **Normalized_TyreLife** and **Cumulative_Degradation** are also strong predictors.
4. **RaceProgress**, **Year**, and **Stint** show moderate correlations.
5. **Position**, **LapTime**, **LapTime_Delta**, and **PitStop** have very weak correlations.

The overall median correlation is low (Pearson: 0.072, Spearman: 0.037), indicating weak relationships overall. This is typical for this kind of data and suggests that more sophisticated modeling (e.g., tree-based methods) may be needed to capture non-linear relationships.

Note: The `Compound` column has 66 missing values and is categorical, so it is not included in the numeric correlation analysis. It will be explored in later notebook steps.

In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("data/f1_strategy_dataset_v4.csv")
print("Dataset shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nTarget distribution:")
print(df["PitNextLap"].value_counts())
print("\nMissing values per column:")
print(df.isna().sum())

Dataset shape: (101371, 16)

Column dtypes:
Driver                     object
LapNumber                   int64
Compound                   object
Stint                       int64
TyreLife                  float64
Position                    int64
LapTime (s)               float64
Race                       object
Year                        int64
LapTime_Delta             float64
Cumulative_Degradation    float64
PitStop                     int64
PitNextLap                  int64
RaceProgress              float64
Normalized_TyreLife       float64
Position_Change           float64
dtype: object

Target distribution:
PitNextLap
0    75542
1    25829
Name: count, dtype: int64

Missing values per column:
Driver                     0
LapNumber                  0
Compound                  66
Stint                      0
TyreLife                   0
Position                   0
LapTime (s)                0
Race                       0
Year                       0
LapTime_Delta              0

In [2]:
target = "PitNextLap"
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(target)
y = df[target].to_numpy(dtype=float)
y_centered = y - y.mean()
y_var = (y_centered * y_centered).sum()
X = df[numeric_cols].to_numpy(dtype=float)
X_centered = X - X.mean(axis=0)
pearson_corr = np.array([float((X_centered[:, i] * y_centered).sum() / np.sqrt((X_centered[:, i] ** 2).sum() * y_var)) for i in range(X.shape[1])])
from scipy.stats import rankdata
ranks = rankdata(X, axis=0)
target_ranks = rankdata(y)
target_ranks_centered = target_ranks - target_ranks.mean()
target_ranks_var = (target_ranks_centered * target_ranks_centered).sum()
spearman_corr = np.array([float((ranks[:, i] * target_ranks_centered).sum() / np.sqrt((ranks[:, i] ** 2).sum() * target_ranks_var)) for i in range(X.shape[1])])
pearson_df = pd.Series(pearson_corr, index=numeric_cols).abs().sort_values(ascending=False)
spearman_df = pd.Series(spearman_corr, index=numeric_cols).abs().sort_values(ascending=False)
print("=== Pearson Correlation (absolute) ===")
print(pearson_df.round(3))
print("\n=== Spearman Correlation (absolute) ===")
print(spearman_df.round(3))
print("\nMedian Pearson:", round(float(np.median(pearson_corr)), 3))
print("Median Spearman:", round(float(np.median(spearman_corr)), 3))

=== Pearson Correlation (absolute) ===
TyreLife                  0.266
LapNumber                 0.160
Normalized_TyreLife       0.125
Cumulative_Degradation    0.122
RaceProgress              0.099
Year                      0.097
Stint                     0.087
Position_Change           0.058
Position                  0.032
LapTime (s)               0.010
PitStop                   0.005
LapTime_Delta             0.004
dtype: float64

=== Spearman Correlation (absolute) ===
TyreLife                  0.139
LapNumber                 0.084
Normalized_TyreLife       0.073
Stint                     0.064
Cumulative_Degradation    0.058
RaceProgress              0.057
Year                      0.050
Position_Change           0.024
Position                  0.016
LapTime (s)               0.006
LapTime_Delta             0.003
PitStop                   0.002
dtype: float64

Median Pearson: 0.072
Median Spearman: 0.037
